# Notebook 13 · Sensitivity to the annotation protocol of CT-ICH (added after the registered analysis)

**Question:** how much of the decline on CT-ICH comes from the way it was annotated? In CT-ICH, every connected
component of the segmentation mask became a reference box, so a fragmented haemorrhage yields several small
boxes; in BHX, readers drew one box around each haemorrhage.

**What it does:** re-scores the stored predictions against four versions of the reference standard:

1. **original** (as in the article);
2. **merged per slice:** all the boxes of a slice become one box;
3. **merged by proximity:** boxes within 16 px (about 8 mm) of each other become one box;
4. **without small boxes:** boxes below 32 × 32 px are left out, together with the detections that fall on them
   (as in a COCO evaluation restricted to an area range).

The same versions are applied to the CQ500 validation boxes as a control: if merging also changes the score
much on CQ500, the effect is not specific to CT-ICH. On the test set, 95% intervals by bootstrap over
examinations (10,000 resamples, seed 20261030, as in notebook 12).

Nothing is trained or predicted again, and the registered results are not touched (a separate file is written).

**Before running:** the `src/ichdet` folder, version 0.6.0, on the Drive. No GPU is needed (*Runtime → Change
runtime type → CPU*). **Time:** about 30 to 60 minutes (almost all of it in the bootstrap).

In [ ]:
# 1. Drive and code
import json, os, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])

sys.path.insert(0, str(RAIZ / 'codigo'))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
PRIMEIRA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'
CORRECAO = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'
SAIDA = CORRECAO / 'exploratorio'
SAIDA.mkdir(exist_ok=True)

import numpy as np
import ichdet
from ichdet import data_io, experiment, exploratory as ex
assert ichdet.__version__ >= '0.6.0', f'ichdet {ichdet.__version__}: copie de novo a pasta src/ichdet do repositório'
print('ichdet', ichdet.__version__)

In [ ]:
# 2. Reference boxes (read from inside the zip files) and stored predictions
A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
BRACOS = (A, B, C)
PASTA = {A: CORRECAO, B: PRIMEIRA, C: PRIMEIRA}
K = experiment.FOLDS

def referencias(arquivo_zip, lista):
    with zipfile.ZipFile(PREPARO / arquivo_zip) as z:
        entradas = json.loads(z.read(lista))['validation']
    return {data_io.image_key(e['image']): (e['exam'], np.asarray(e['box'], dtype=float).reshape(-1, 4)) for e in entradas}

REF = {k: referencias('cq500_512.zip', f'monai/fold{k}.json') for k in K}
REF_TESTE = referencias('ctich_512.zip', 'monai/external.json')
PRED = {(k, b): data_io.load_predictions(experiment.job_dir(PASTA[b], k, b) / 'pred_validation.csv') for k in K for b in BRACOS}
PRED_TESTE = {(k, b): data_io.load_predictions(experiment.job_dir(PASTA[b], k, b) / 'pred_test.csv') for k in K for b in BRACOS}
print(f"validação: {sum(len(v) for v in REF.values())} cortes; teste: {len(REF_TESTE)} cortes")

In [ ]:
# 3. The four versions of the reference (and of the predictions, in the version without small boxes)
VERSOES = ('original', 'juntas_por_corte', 'juntas_por_proximidade', 'sem_pequenas')
GAP = 16                                             # px; about 8 mm at 0.488 mm per pixel

def versao(nome, ref, pred):
    if nome == 'original':
        return ref, pred
    if nome == 'juntas_por_corte':
        return ex.merge_reference_boxes(ref), pred
    if nome == 'juntas_por_proximidade':
        return ex.merge_reference_boxes(ref, gap=GAP), pred
    return ex.ignore_small_references(ref, pred)

def n_caixas(ref):
    return int(sum(len(b) for _, b in ref.values()))

caixas = {'validation': {}, 'test': {}}
for v in VERSOES:
    caixas['test'][v] = n_caixas(versao(v, REF_TESTE, {})[0])
    caixas['validation'][v] = sum(n_caixas(versao(v, REF[k], {})[0]) for k in K)
print('caixas de referência por versão:', json.dumps(caixas, indent=1))

In [ ]:
# 4. AP@0.5 of each arm in each version (mean of the 5 models), in validation (control) and on the test set
def tabelas(v, refs, preds):
    pares = [versao(v, r, p) for r, p in zip(refs, preds)]
    return ex.build_tables([g for g, _ in pares], [p for _, p in pares])

T_VAL = {(v, b): tabelas(v, [REF[k] for k in K], [PRED[(k, b)] for k in K]) for v in VERSOES for b in BRACOS}
T_TESTE = {(v, b): tabelas(v, [REF_TESTE] * len(K), [PRED_TESTE[(k, b)] for k in K]) for v in VERSOES for b in BRACOS}

# check: in the original version, AP@0.5 must equal the registered results
registrado = json.loads((CORRECAO / 'results_test.json').read_text())['A_vs_B']
for b, chave in ((A, 'A'), (B, 'B')):
    assert np.isclose(np.mean([t.ap50() for t in T_TESTE[('original', b)]]), registrado[chave]['ap50']), b
print('AP@0.5 original confere com os resultados registrados\n')

ap = {s: {v: {b: float(np.mean([t.ap50() for t in T[(v, b)]])) for b in BRACOS} for v in VERSOES}
      for s, T in (('validation', T_VAL), ('test', T_TESTE))}
print(f"{'versão':24s} " + ' '.join(f'{b:>16s}' for b in BRACOS))
for s in ('validation', 'test'):
    print(s)
    for v in VERSOES:
        print(f"  {v:22s} " + ' '.join(f"{ap[s][v][b]:16.3f}" for b in BRACOS))

In [ ]:
# 5. Test set: 95% intervals (bootstrap over exams; same draws for the three arms) in each version
N, SEMENTE = 10_000, 20261030
n = len(T_TESTE[('original', A)][0].exams)
teste = {}
for v in VERSOES:
    inicio = time.time()
    T = {b: T_TESTE[(v, b)] for b in BRACOS}
    teste[v] = ex.summarise(T, {b: ex.thresholds_1fp(T[b]) for b in BRACOS}, lambda w: [w[0]] * len(K), [n],
                            N, SEMENTE, reference_arm=B)
    d = teste[v]['differences'][f'{A}_minus_{B}']['ap50']
    print(f"{v:24s} RetinaNet − YOLO sem mosaico: {d['value']:+.3f} (IC 95% {d['ci95'][0]:+.3f} a {d['ci95'][1]:+.3f}); "
          f"{time.time() - inicio:.0f} s")

In [ ]:
# 6. Drop from validation to test in each version, and output file
queda = {v: {b: ap['validation']['original'][b] - ap['test'][v][b] for b in BRACOS} for v in VERSOES}
for v in VERSOES:
    print(f"{v:24s} queda em relação à validação original: " + '  '.join(f"{b} {queda[v][b]:.3f}" for b in BRACOS))

resultado = {
    'note': ('Sensitivity of the results to the annotation protocol of CT-ICH (co-author comment of 20 Nov 2026); '
             'exploratory, not in the registered plan.'),
    'computed': time.strftime('%Y-%m-%d %H:%M:%S'), 'ichdet_version': ichdet.__version__,
    'versions': {'original': 'reference boxes as in the article',
                 'juntas_por_corte': 'all reference boxes of a slice merged into one enclosing box',
                 'juntas_por_proximidade': f'reference boxes within {GAP} px of each other merged into one box',
                 'sem_pequenas': 'reference boxes below 32 x 32 px left out, with the detections that match them (IoU >= 0.5)'},
    'reference_boxes': caixas, 'ap50': ap, 'drop_from_original_validation': queda, 'test': teste,
}
(SAIDA / 'annotation_sensitivity.json').write_text(json.dumps(resultado, indent=1, default=float))
print('gravado em', SAIDA / 'annotation_sensitivity.json')

## After running

1. Save the executed notebook (*File → Download → .ipynb*).
2. Download `annotation_sensitivity.json` from `dados/tc/experimento/correcao_retinanet/exploratorio/` on the
   Drive; it is the file `data/correcao_retinanet/annotation_sensitivity.json` of this repository.